In [ ]:
from pathlib import Path
from pathlib import Path
from FramePrep import select_frames, resize_fragments
from pytorch_model import SiameseNetwork
from torch.utils.data import DataLoader
import torch.nn as nn
from pathlib import Path
import pandas as pd
from pathlib import Path
import random
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

ann_file = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation/troy_achilles_and_hector.txt")

with open(ann_file, "r") as f:
    for i in range(10):
        print(f.readline())

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:12,00:01:26,00:04:38

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,548e9ff21d5a54cf621986dc6047abf8db12e8b7.flv,00:00:00,00:03:12,00:01:08,00:04:13

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,6280e17346f8c34c231180e8d659f804d111fa6b.mp4,00:00:01,00:03:08,00:00:26,00:01:32

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,6fe097a963673b26c62f6ff6d6151d383c194b9d.flv,00:00:00,00:03:06,00:01:25,00:04:24

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,7fdf8f5e4fb91b1af5b016242ad70fd0d74abb42.flv,00:00:00,00:03:06,00:02:55,00:05:04

0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,868f1ff130f7a0215de9abfacca746f96c4cd85d.flv,00:00:00,00:03:07,00:

In [23]:
from pathlib import Path

ann_file = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation/troy_achilles_and_hector.txt")

with open(ann_file, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        print("RAW:", repr(line))
        print("split():", line.split())
        print("split tab:", line.split("\t"))
        print("-" * 50)
        if i == 4:
            break

RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12'
split(): ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12']
split tab: ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12']
--------------------------------------------------
RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12'
split(): ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12']
split tab: ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12']
--------------------------------------------------
RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d

In [ ]:
from pathlib import Path
import pandas as pd

ann_dir = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation")

records = []

for ann_file in ann_dir.glob("*.txt"):

    with open(ann_file, "r") as f:

        for line in f:

            parts = line.strip().split(",")

            if len(parts) != 6:
                continue

            video1, video2, start1, end1, start2, end2 = parts

            records.append({
                "video1": video1,
                "video2": video2,
                "start1": start1,
                "end1": end1,
                "start2": start2,
                "end2": end2,
            })

df = pd.DataFrame(records)



9236


In [ ]:

annotation_dir = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation")

def hms_to_seconds(s: str) -> float:
    h, m, sec = s.split(":")
    return int(h) * 3600 + int(m) * 60 + float(sec)

records = []

for ann_file in annotation_dir.glob("*.txt"):
    with open(ann_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) != 6:
                continue

            video1, video2, start1, end1, start2, end2 = parts

            records.append({
                "group": ann_file.stem,
                "video1": video1,
                "video2": video2,
                "start1": start1,
                "end1": end1,
                "start2": start2,
                "end2": end2,
                "start1_sec": hms_to_seconds(start1),
                "end1_sec": hms_to_seconds(end1),
                "start2_sec": hms_to_seconds(start2),
                "end2_sec": hms_to_seconds(end2),
            })

labels_df = pd.DataFrame(records)

print(labels_df.head())
print("Total segment pairs:", len(labels_df))
print("Unique video pairs:", labels_df[["video1", "video2"]].drop_duplicates().shape[0])

                 group                                        video1  \
0  baggio_penalty_1994  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
1  baggio_penalty_1994  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
2  baggio_penalty_1994  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
3  baggio_penalty_1994  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
4  baggio_penalty_1994  3504e360accbaccb1580befbb441f1019664c2bb.mp4   

                                         video2    start1      end1    start2  \
0  3504e360accbaccb1580befbb441f1019664c2bb.mp4  00:00:00  00:00:09  00:00:00   
1  37b31d607d31a47d347b15dae2b8aa63e57861eb.flv  00:00:03  00:00:05  00:00:33   
2  37b31d607d31a47d347b15dae2b8aa63e57861eb.flv  00:00:03  00:00:05  00:00:41   
3  458db5aa227ae49ceb8bc1bed5f9cf5b4bed63f2.flv  00:00:00  00:00:07  00:00:05   
4  5c5714c0a56fd2a96f99db2f59b0d03659d77cdf.flv  00:00:00  00:00:04  00:00:06   

       end2  start1_sec  end1_sec  start2_sec  end2_sec  
0  00:00:09         0.

In [26]:
labels_df = labels_df[labels_df["video1"] != labels_df["video2"]].reset_index(drop=True)

In [27]:
video_pairs_df = labels_df[["video1", "video2"]].drop_duplicates().reset_index(drop=True)

positive_pairs = set()
for _, row in video_pairs_df.iterrows():
    a, b = row["video1"], row["video2"]
    positive_pairs.add((a, b))
    positive_pairs.add((b, a))

In [28]:
def is_positive(video_a, video_b):
    return (video_a, video_b) in positive_pairs

In [29]:


vcdb_root = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB")

video_index = {}
for p in vcdb_root.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".mp4", ".flv", ".avi", ".mov", ".mkv"}:
        video_index[p.name] = str(p)

print("Indexed videos:", len(video_index))
print(list(video_index.items())[:5])

Indexed videos: 574
[('9b15a52652373497a10b8915b8cc025967962b72.flv', '/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/core_dataset/troy_achilles_and_hector/9b15a52652373497a10b8915b8cc025967962b72.flv'), ('9c5fde27983574ca7840da09d40c40bd20820957.flv', '/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/core_dataset/troy_achilles_and_hector/9c5fde27983574ca7840da09d40c40bd20820957.flv'), ('9f7862bcee961fdec039a3f287ad7c8351cfbed9.flv', '/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/core_dataset/troy_achilles_and_hector/9f7862bcee961fdec039a3f287ad7c8351cfbed9.flv'), ('a0fc0a97772388012c3d2062b36086d2cfb2bc82.flv', '/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/core_dataset/troy_achilles_and_hector/a0fc0a97772388012c3d2062b36086d2cfb2bc82.flv'), ('a15a43f0cf9ce7eab781706a82ee0027d53aae4f.flv', '/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/core_dataset/troy_achilles_and_hector/a15a43f0cf9ce7eab781706a82ee0027d53aae4f.flv')]


In [30]:
labels_df["video1_path"] = labels_df["video1"].map(video_index)
labels_df["video2_path"] = labels_df["video2"].map(video_index)

print(labels_df[["video1", "video1_path", "video2", "video2_path"]].head())

                                         video1  \
0  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
1  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
2  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
3  3504e360accbaccb1580befbb441f1019664c2bb.mp4   
4  3504e360accbaccb1580befbb441f1019664c2bb.mp4   

                                         video1_path  \
0  /Volumes/NO NAME/DuplicateDetectionThesis/VCDB...   
1  /Volumes/NO NAME/DuplicateDetectionThesis/VCDB...   
2  /Volumes/NO NAME/DuplicateDetectionThesis/VCDB...   
3  /Volumes/NO NAME/DuplicateDetectionThesis/VCDB...   
4  /Volumes/NO NAME/DuplicateDetectionThesis/VCDB...   

                                         video2  \
0  37b31d607d31a47d347b15dae2b8aa63e57861eb.flv   
1  37b31d607d31a47d347b15dae2b8aa63e57861eb.flv   
2  458db5aa227ae49ceb8bc1bed5f9cf5b4bed63f2.flv   
3  5c5714c0a56fd2a96f99db2f59b0d03659d77cdf.flv   
4  67824b87c0698c499acad123b7498ae17f97bf6d.flv   

                                         video2_p

In [31]:
video_pairs_df = labels_df[["video1", "video2"]].drop_duplicates()

positive_pairs = set()
for _, row in video_pairs_df.iterrows():
    a, b = row["video1"], row["video2"]
    positive_pairs.add((a, b))
    positive_pairs.add((b, a))

In [ ]:
from pathlib import Path

ann_file = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation/troy_achilles_and_hector.txt")

with open(ann_file, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        print("RAW:", repr(line))
        print("split():", line.split())
        print("split tab:", line.split("\t"))
        print("-" * 50)
        if i == 4:
            break

RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12'
split(): ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12']
split tab: ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,00:00:00,00:03:12,00:00:00,00:03:12']
--------------------------------------------------
RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12'
split(): ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12']
split tab: ['0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d1a32115ea3457867128.flv,00:00:00,00:03:04,00:00:00,00:03:12']
--------------------------------------------------
RAW: '0a6a553c5d2d7eb7409eb580e7da01f60e3176f1.flv,0b3f9e88e5ab73e19dc4d

In [ ]:


vcdb_root = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB")

video_index = {}

for p in vcdb_root.rglob("*"):
    if p.suffix.lower() in [".mp4", ".flv"]:
        video_index[p.name] = str(p)

print("Videos indexed:", len(video_index))

Videos indexed: 574


In [33]:
all_videos = list(video_index.keys())

In [ ]:


def sample_frames(video_path, num_frames=8):
    
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count == 0:
        cap.release()
        return None

    indices = np.linspace(0, frame_count-1, num_frames).astype(int)

    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv2.resize(frame, (224,224))
        frame = frame[:,:,::-1]  # BGR → RGB
        frames.append(frame)

    cap.release()

    frames = np.stack(frames)
    frames = frames / 255.0

    return frames

In [35]:
import torch
from torch.utils.data import Dataset
import random

class VCDBSiameseDataset(Dataset):

    def __init__(self, video_index, positive_pairs):

        self.video_index = video_index
        self.video_names = list(video_index.keys())
        self.positive_pairs = positive_pairs

        self.pos_list = list(set((a,b) for a,b in positive_pairs))

    def __len__(self):
        return len(self.pos_list)

    def __getitem__(self, idx):

        video1, video2 = self.pos_list[idx]

        path1 = self.video_index[video1]
        path2 = self.video_index[video2]

        frames1 = sample_frames(path1)
        frames2 = sample_frames(path2)

        label = 1

        # randomly replace with negative pair half the time
        if random.random() < 0.5:

            neg_video = random.choice(self.video_names)

            while (video1, neg_video) in self.positive_pairs:
                neg_video = random.choice(self.video_names)

            path2 = self.video_index[neg_video]
            frames2 = sample_frames(path2)

            label = 0

        frames1 = torch.tensor(frames1).permute(0,3,1,2).float()
        frames2 = torch.tensor(frames2).permute(0,3,1,2).float()

        return frames1, frames2, torch.tensor(label).float()

In [36]:
from torch.utils.data import DataLoader

dataset = VCDBSiameseDataset(video_index, positive_pairs)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4
)

In [ ]:

class VCDBFragmentPairDataset(Dataset):
    def __init__(self, labels_df, video_index, target_size=(32, 32), len_frames=15, negative_prob=0.5):
        self.labels_df = labels_df.reset_index(drop=True)
        self.video_index = video_index
        self.target_size = target_size
        self.len_frames = len_frames
        self.negative_prob = negative_prob

        self.pos_pairs = list(
            set(
                (row.video1, row.video2)
                for _, row in self.labels_df.iterrows()
                if row.video1 != row.video2
            )
        )

        self.video_names = list(video_index.keys())

        self.positive_lookup = set()
        for a, b in self.pos_pairs:
            self.positive_lookup.add((a, b))
            self.positive_lookup.add((b, a))

    def __len__(self):
        return len(self.pos_pairs)

    def load_video_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()
        return np.array(frames)

    def get_fragments(self, video_name):
        video_path = self.video_index[video_name]
        frames = self.load_video_frames(video_path)

        if len(frames) < 2:
            return None

        fragments = select_frames(frames, len_frames=self.len_frames)
        if len(fragments) == 0:
            return None

        fragments = resize_fragments(fragments, target_size=self.target_size)
        return fragments

    def sample_negative_video(self, anchor):
        while True:
            candidate = random.choice(self.video_names)
            if candidate != anchor and (anchor, candidate) not in self.positive_lookup:
                return candidate

    def __getitem__(self, idx):
        video1, video2 = self.pos_pairs[idx]

        label = 1.0
        if random.random() < self.negative_prob:
            video2 = self.sample_negative_video(video1)
            label = 0.0

        frags1 = self.get_fragments(video1)
        frags2 = self.get_fragments(video2)

        if frags1 is None or frags2 is None:
            new_idx = random.randint(0, len(self) - 1)
            return self.__getitem__(new_idx)

        frag1 = frags1[random.randint(0, len(frags1) - 1)]
        frag2 = frags2[random.randint(0, len(frags2) - 1)]

        frag1 = torch.tensor(frag1, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        frag2 = torch.tensor(frag2, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        label = torch.tensor(label, dtype=torch.float32)

        return frag1, frag2, label

In [ ]:
model = SiameseNetwork()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=1e-3
)

: 

In [ ]:


pairs = df[['video1','video2']].drop_duplicates()

train_pairs, temp = train_test_split(pairs, test_size=0.3, random_state=42)

val_pairs, test_pairs = train_test_split(temp, test_size=0.5)

In [ ]:


dataset = VCDBFragmentPairDataset(labels_df, video_index, target_size=(32, 32), len_frames=15)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

criterion = nn.BCEWithLogitsLoss()

for epoch in range(10):
    model.train()
    total_loss = 0.0

    for x1, x2, y in loader:
        optimizer.zero_grad()

        logits, z1, z2 = model(x1, x2)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss = {total_loss / len(loader):.4f}")

[flv @ 0x15d427280] Packet mismatch 1024 1050 26933818
[flv @ 0x14e562e50] Packet mismatch 1024 1050 26933818


In [1]:
import csv
from pathlib import Path
import numpy as np

ANNOTATION_DIR = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation")

random_ann = np.random.choice(list(ANNOTATION_DIR.glob("*.txt")))
print("FILE:", random_ann)

with open(random_ann, "r", encoding="latin-1") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(i, row, "len =", len(row))
        if i > 20:
            break

FILE: /Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation/baggio_penalty_1994.txt
0 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '3504e360accbaccb1580befbb441f1019664c2bb.mp4', '00:00:00', '00:00:09', '00:00:00', '00:00:09'] len = 6
1 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '37b31d607d31a47d347b15dae2b8aa63e57861eb.flv', '00:00:03', '00:00:05', '00:00:33', '00:00:35'] len = 6
2 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '37b31d607d31a47d347b15dae2b8aa63e57861eb.flv', '00:00:03', '00:00:05', '00:00:41', '00:00:45'] len = 6
3 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '458db5aa227ae49ceb8bc1bed5f9cf5b4bed63f2.flv', '00:00:00', '00:00:07', '00:00:05', '00:00:12'] len = 6
4 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '5c5714c0a56fd2a96f99db2f59b0d03659d77cdf.flv', '00:00:00', '00:00:04', '00:00:06', '00:00:10'] len = 6
5 ['3504e360accbaccb1580befbb441f1019664c2bb.mp4', '67824b87c0698c499acad123b7498ae17f97bf6d.flv', '00:00:00', '00:00:06', '00:00:00', '0

In [2]:
import csv
from pathlib import Path

ANNOTATION_DIR = Path("/Volumes/NO NAME/DuplicateDetectionThesis/VCDB/annotation")

bad_rows = []

for ann_file in ANNOTATION_DIR.glob("*.txt"):
    try:
        with open(ann_file, "r", encoding="latin-1") as f:
            for i, row in enumerate(csv.reader(f)):
                if len(row) < 6:
                    bad_rows.append((ann_file.name, i, row, len(row)))
    except Exception as e:
        print("ERROR reading", ann_file.name, e)

print("Number of bad rows:", len(bad_rows))
for item in bad_rows[:50]:
    print(item)

Number of bad rows: 1
('._baggio_penalty_1994.txt', 0, ['\x00\x05\x16\x07\x00\x02\x00\x00Mac OS X        \x00\x02\x00\x00\x00\t\x00\x00\x002\x00\x00\x0e°\x00\x00\x00\x02\x00\x00\x0eâ\x00\x00\x01\x1e\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00ATTRÿÿÿ¹\x00\x00\x0eâ\x00\x00\x00\xa0\x00\x00\x00\x10\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\xa0\x00\x00\x00\x10\x00\x00\x1acom.apple.lastuseddate#PS\x00\x00\x00\x00aé®i\x00\x00\x00\x00\x07Á#)\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

In [1]:
# Count descriptors (fragments) per video

import numpy as np
from pathlib import Path

DESCR_DIR = Path("/Volumes/LaCie/DuplicateDetectionThesis/VCDB/descriptors")


def load_descriptor_file(path):
    obj = np.load(path, allow_pickle=True).item()
    return obj["descriptors"]


results = []

for video_folder in sorted(DESCR_DIR.glob("*")):
    if not video_folder.is_dir():
        continue

    video_id = video_folder.name
    total_fragments = 0
    num_files = 0

    for descriptor_file in video_folder.glob("*.npy"):
        try:
            desc = load_descriptor_file(descriptor_file)
            num_files += 1

            # Case 1: single fragment
            if desc.ndim == 2:
                total_fragments += 1

            # Case 2: multiple fragments
            elif desc.ndim == 3:
                total_fragments += desc.shape[0]

            else:
                print(f"[WARN] Unexpected shape {desc.shape} in {descriptor_file}")

        except Exception as e:
            print(f"[ERROR] {descriptor_file}: {e}")

    results.append((video_id, total_fragments, num_files))

# Sort by number of fragments (descending)
results.sort(key=lambda x: x[1], reverse=True)

print("\n=== Descriptor count per video ===\n")
for video_id, fragments, files in results:
    print(f"{video_id:60s} | fragments: {fragments:6d} | files: {files}")

print("\nSummary:")
print(f"Total videos: {len(results)}")
print(f"Total fragments: {sum(r[1] for r in results)}")


=== Descriptor count per video ===

osama_bin_laden_is_dead_obama_speech_at_white_house          | fragments:   6773 | files: 36
scent_of_woman_tango                                         | fragments:   3606 | files: 36
saving_private_ryan_omaha_beach                              | fragments:   3084 | files: 24
the_last_samurai_last_battle                                 | fragments:   2573 | files: 15
troy_achilles_and_hector                                     | fragments:   2360 | files: 30
t-mac_13_points_in_35_seconds                                | fragments:   2121 | files: 24
president_obama_takes_oath                                   | fragments:   2027 | files: 31
titanic_fly_scene                                            | fragments:   1895 | files: 35
obama_kicks_door                                             | fragments:   1806 | files: 17
the_pursuit_of_happyness_-_job_interview                     | fragments:   1306 | files: 13
endless_love                     

In [1]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd

TEST_CSV = Path("test_pairs.csv")

df = pd.read_csv(TEST_CSV)
print("Columns:", df.columns.tolist())
print("Rows:", len(df))
print(df.head(10).to_string(index=False))

Columns: ['pair_id', 'x1_path', 'x2_path', 'label', 'video1', 'video2', 'pair_type', 'start1', 'end1', 'start2', 'end2']
Rows: 2739
 pair_id                                                                          x1_path                                                                          x2_path  label                                   video1                                   video2     pair_type   start1     end1       start2         end2
    1310 /Volumes/NO NAME/DuplicateDetectionThesis/VCDB/positive_pairs/pair_001310_x1.npy /Volumes/NO NAME/DuplicateDetectionThesis/VCDB/positive_pairs/pair_001310_x2.npy    1.0 8c207ab5a07d12d37dcb666ee8b69c3357bc07fb 9c1d7e91288e07083e4456f8fe6b771aba039ed1      positive 00:01:42 00:01:47     00:00:24     00:00:30
    7340 /Volumes/NO NAME/DuplicateDetectionThesis/VCDB/positive_pairs/pair_007340_x1.npy /Volumes/NO NAME/DuplicateDetectionThesis/VCDB/positive_pairs/pair_007340_x2.npy    1.0 9ad3cc564c4f1b52ce076068e3830a70b9abccbf c4365f3b67428